In [ ]:
# import kagglehub
# path = kagglehub.dataset_download("muhammadabdulsami/massive-skin-disease-balanced-dataset")

In [13]:
path="/Volumes/CrucialX9/Project/Data Science/Image Based/Skin Disease/archive/balanced_dataset/balanced_dataset"

In [14]:
import os

dataset_path = path  # from kagglehub
class_names = sorted([
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
])

print(class_names)
print("Number of classes:", len(class_names))

['Acne And Rosacea Photos', 'Actinic Keratosis Basal Cell Carcinoma And Other Malignant Lesions', 'Atopic Dermatitis Photos', 'Ba  Cellulitis', 'Ba Impetigo', 'Benign', 'Bullous Disease Photos', 'Cellulitis Impetigo And Other Bacterial Infections', 'Eczema Photos', 'Exanthems And Drug Eruptions', 'Fu Athlete Foot', 'Fu Nail Fungus', 'Fu Ringworm', 'Hair Loss Photos Alopecia And Other Hair Diseases', 'Heathy', 'Herpes Hpv And Other Stds Photos', 'Light Diseases And Disorders Of Pigmentation', 'Lupus And Other Connective Tissue Diseases', 'Malignant', 'Melanoma Skin Cancer Nevi And Moles', 'Nail Fungus And Other Nail Disease', 'Pa Cutaneous Larva Migrans', 'Poison Ivy Photos And Other Contact Dermatitis', 'Psoriasis Pictures Lichen Planus And Related Diseases', 'Rashes', 'Scabies Lyme Disease And Other Infestations And Bites', 'Seborrheic Keratoses And Other Benign Tumors', 'Systemic Disease', 'Tinea Ringworm Candidiasis And Other Fungal Infections', 'Urticaria Hives', 'Vascular Tumors',

In [15]:
selected_classes = [
    'Acne And Rosacea Photos',
    'Actinic Keratosis Basal Cell Carcinoma And Other Malignant Lesions',
    'Atopic Dermatitis Photos',
    'Ba  Cellulitis',  # <-- TWO spaces (fix)
    'Ba Impetigo',
    'Benign',
    'Bullous Disease Photos',
    'Cellulitis Impetigo And Other Bacterial Infections',
    'Eczema Photos',
    'Exanthems And Drug Eruptions',
    'Fu Athlete Foot',
    'Fu Nail Fungus',
    'Fu Ringworm',
    'Hair Loss Photos Alopecia And Other Hair Diseases',
    'Herpes Hpv And Other Stds Photos',
    'Light Diseases And Disorders Of Pigmentation',
    'Lupus And Other Connective Tissue Diseases',
    'Malignant',
    'Melanoma Skin Cancer Nevi And Moles',
    'Rashes',
    'Heathy'  # <-- keep typo
]

In [16]:
import tensorflow as tf
import os
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    class_names=selected_classes  # THIS is key
)

Found 158520 files belonging to 21 classes.


In [17]:
import tensorflow as tf

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    class_names=selected_classes,
    image_size=(240, 240),
    batch_size=16
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    class_names=selected_classes,
    image_size=(240, 240),
    batch_size=16
)

Found 158520 files belonging to 21 classes.
Using 126816 files for training.
Found 158520 files belonging to 21 classes.
Using 31704 files for validation.


In [18]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(200).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [19]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [20]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(240, 240, 3)
)

base_model.trainable = False  # freeze initially

In [21]:
num_classes = len(selected_classes)

inputs = tf.keras.Input(shape=(240, 240, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

In [22]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [23]:
import tensorflow as tf

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

Num GPUs Available: 0
[]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
 145/7926 ━━━━━━━━━━━━━━━━━━━━ 40:24 312ms/step - accuracy: 0.1856 - loss: 3.2950

In [ ]:
base_model.trainable = True

# Freeze early layers, train deeper ones
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
model.save("skin_disease_model.h5")